VIKTOR LENAC OPTIMATION PLAN AND MODEL

In [25]:
#Install Gurobi
%pip install gurobipy


In [26]:
import gurobipy as gp
from gurobipy import GRB

DEFINING THE PARAMETERS FROM THE CASESTUDY

In [27]:
#contribution per contract ($)
contribution = {
    'Repair':          520171,
    'Conversion':     5014710,
    'Offshore':       3098844,
    'Newbuilding':    1600150
}


In [28]:
#maximum contract available
max_contracts = {
    'Repair':       40,
    'Conversion':    4,
    'Offshore':      2,
    'Newbuilding':   2
}




In [29]:
#resource usage per contract
steel_usage = {
    'Repair':      170,
    'Conversion': 1100,
    'Offshore':    418,
    'Newbuilding': 300
}

corrosion_usage = {
    'Repair':      5700,
    'Conversion': 48000,
    'Offshore':    4200,
    'Newbuilding':10000
}

dock_usage = {
    'Repair':      9,
    'Conversion': 83,
    'Offshore':    0,
    'Newbuilding':20
}

labor_usage = {
    'Repair':      47645,
    'Conversion': 731000,
    'Offshore':   170609,
    'Newbuilding':100426
}


In [30]:
#resource capabilities / constraints
steel_capacity     = 5500
corrosion_capacity = 500000
dock_capacity      = 700
labor_capacity     = 4000000

#profitability target
fixed_costs_target = 30000000


CREATING THE MODEL

In [31]:
# Creating the optimization model

model = gp.Model('Viktor Lenac Production Planning')

ADDING THE DECISION VARIABLES

In [32]:
# x[j] = number of contracts of type j to accept
x = model.addVars(contribution.keys(), vtype=GRB.INTEGER, name='contracts')

ADDING THE CONSTRAINTS

In [33]:
#Contract availability constraints
for j in contribution.keys():
    model.addConstr(x[j] <= max_contracts[j], name=f'max_{j}')

#Non-negativity constraints
for j in contribution.keys():
    model.addConstr(x[j] >= 0, name=f'min_{j}')


In [34]:
# Resource constraints
# Steel: total steel used cannot exceed 5,500 tons
model.addConstr(
    gp.quicksum(steel_usage[j] * x[j] for j in contribution.keys())
    <= steel_capacity,
    name='Steel'
)

# Corrosion Protection: total painting capacity cannot exceed 500,000 m²
model.addConstr(
    gp.quicksum(corrosion_usage[j] * x[j] for j in contribution.keys())
    <= corrosion_capacity,
    name='Corrosion_Protection'
)

# Dock Days: total dock days used cannot exceed 700 days
model.addConstr(
    gp.quicksum(dock_usage[j] * x[j] for j in contribution.keys())
    <= dock_capacity,
    name='Dock_Days'
)

# Labor Hours: total labor hours used cannot exceed 4,000,000 hours
model.addConstr(
    gp.quicksum(labor_usage[j] * x[j] for j in contribution.keys())
    <= labor_capacity,
    name='Labor_Hours'
)

<gurobi.Constr *Awaiting Model Update*>

ADDING THE OBJECTIVE FUNCTION

In [35]:
model.setObjective(
    gp.quicksum(contribution[j] * x[j] for j in contribution.keys()),
    GRB.MAXIMIZE
)


MODEL UPDATE AND WRITE

In [36]:
model.update()
model.write('Viktor Lenac Production Planning.lp')
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 12 rows, 4 columns and 23 nonzeros (Max)
Model fingerprint: 0x95aae4af
Model has 4 linear objective coefficients
Variable types: 0 continuous, 4 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+05]
  Objective range  [5e+05, 5e+06]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 4e+06]

Found heuristic solution: objective 1.664547e+07
Presolve removed 11 rows and 0 columns
Presolve time: 0.00s
Presolved: 1 rows, 4 columns, 4 nonzeros
Variable types: 0 continuous, 4 integer (0 binary)

Root relaxation: objective 2.792506e+07, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd 

SOLVING THE MODEL

In [38]:
def print_solution(model):
    if model.status != GRB.OPTIMAL:
        print("Model did not find an optimal solution")
    else:
        print(f"Optimal Contribution: ${model.ObjVal:,.0f}")
        print(f"Fixed Costs Target:   $30,000,000")
        print(f"Gap vs Target:        ${model.ObjVal - fixed_costs_target:,.0f}")
        print("=" * 50)

        print("\nOptimal Production Plan:")
        for j in contribution.keys():
            print(f"  {j}: {x[j].X:.0f} contracts")

        print("=" * 50)

        print("\nResource Usage:")
        print(f"  Steel:                {sum(steel_usage[j]     * x[j].X for j in contribution.keys()):,.0f} / {steel_capacity:,.0f} tons")
        print(f"  Corrosion Protection: {sum(corrosion_usage[j] * x[j].X for j in contribution.keys()):,.0f} / {corrosion_capacity:,.0f} m²")
        print(f"  Dock Days:            {sum(dock_usage[j]      * x[j].X for j in contribution.keys()):,.0f} / {dock_capacity:,.0f} days")
        print(f"  Labor Hours:          {sum(labor_usage[j]     * x[j].X for j in contribution.keys()):,.0f} / {labor_capacity:,.0f} hours")
        print("=" * 50)

        print(f"\nCovers Fixed Costs? {'YES' if model.ObjVal >= fixed_costs_target else 'NO'}")

print_solution(model)

Optimal Contribution: $26,776,699
Fixed Costs Target:   $30,000,000
Gap vs Target:        $-3,223,301

Optimal Production Plan:
  Repair: 1 contracts
  Conversion: 4 contracts
  Offshore: 2 contracts
  Newbuilding: -0 contracts

Resource Usage:
  Steel:                5,406 / 5,500 tons
  Corrosion Protection: 206,100 / 500,000 m²
  Dock Days:            341 / 700 days
  Labor Hours:          3,312,863 / 4,000,000 hours

Covers Fixed Costs? NO
